In [1]:
import cv2
import os
import numpy as np
import tensorflow as tf
from PIL import Image
import glob


In [2]:
def extract_frames(video_path, output_folder, num_frames=10):
    os.makedirs(output_folder, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frame_count = 0
    
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        success, image = cap.read()
        if success:
            cv2.imwrite(f"{output_folder}/frame_{frame_count}.jpg", image)
            frame_count += 1
    
    cap.release()
    
# Example usage
extract_frames("data/REAL/ellavthztb.mp4", "test_frames")
# extract_frames("data/FAKE/adylbeequz.mp4", "test_frames")
# extract_frames("imagedataset/test_videos/jiavqbrkyk.mp4", "test_frames")


In [3]:
def preprocess_image(image_path):
    img = Image.open(image_path).convert("RGB")
    img = img.resize((224, 224))  # Adjust according to model input size
    img_array = np.array(img) / 255.0  # Normalize
    return np.expand_dims(img_array, axis=0)

# Example usage
image_tensor = preprocess_image("test_frames/frame_0.jpg")


In [4]:
model = tf.keras.models.load_model("models/vg16.h5")


In [5]:
def predict_frame(image_path):
    img_tensor = preprocess_image(image_path)
    prediction = model.predict(img_tensor)
    print(prediction[0])
    return "FAKE" if prediction[0] > 0.05 else "REAL"  # Assuming binary classification

# Example usage
result = predict_frame("test_frames/frame_9.jpg")
print(f"Prediction: {result}")


1/1 [==============================] - 17s 17s/step
[0.02641892]
Prediction: REAL


In [6]:
frame_files = sorted(glob.glob("test_frames/*.jpg"))
predictions = [predict_frame(frame) for frame in frame_files]
final_decision = "FAKE" if predictions.count("FAKE") > predictions.count("REAL") else "REAL"

print(f"Final Video Prediction: {final_decision}")


1/1 [==============================] - 0s 67ms/step
[0.02896003]
1/1 [==============================] - 0s 32ms/step
[0.02891171]
1/1 [==============================] - 0s 53ms/step
[0.02685265]
1/1 [==============================] - 0s 58ms/step
[0.02957787]
1/1 [==============================] - 0s 34ms/step
[0.02908565]
1/1 [==============================] - 0s 34ms/step
[0.02852843]
1/1 [==============================] - 0s 93ms/step
[0.03288344]
1/1 [==============================] - 0s 51ms/step
[0.02655246]
1/1 [==============================] - 0s 33ms/step
[0.02996833]
1/1 [==============================] - 0s 25ms/step
[0.02641892]
Final Video Prediction: REAL
